# Notebook 4 - Aandrijving, motor en optimalisatie

Deze notebook gebruikt de belastingsresultaten uit `Notebook 3.ipynb` om de praktische aandrijving te dimensioneren. Notebook 3 blijft de bron voor de mechanische belasting: zwaartekracht, wrijving, aandrijfkracht, houdkracht en framebelasting. Notebook 4 vertaalt die resultaten naar een motor-, tandriem/kabel- en remconcept.

Workflow:

1. Run de kinematicanotebook.
2. Run de inertie-only dynamicanotebook.
3. Run `Notebook 3.ipynb` voor de belasting met zwaartekracht en wrijving.
4. Run deze notebook voor aandrijfkeuze, motorkoppel, vermogen, rem, precisie en ontwerpkeuzes.

De gekozen voorkeursarchitectuur is een 24 V DC/BLDC reductiemotor met encoder en rem, onderaan in de mastvoet. Die motor drijft een gesloten tandriem- of kabel/riemlus langs de mast aan. De riem levert alleen de verticale schuifkracht in de `s`-richting. De schuivergeleiding en de mast nemen de grote zijreacties op.


## Setup en data uit Notebook 3

Deze notebook herberekent de inverse dynamica niet. Ze leest `notebook3_gravity_friction_results.npz` en gebruikt daaruit:

- `F_drive_s_total`: totale benodigde aandrijfkracht in positieve `s`-richting.
- `F_hold_s_curve`: statische houdkracht als functie van schuiverpositie.
- `ds`: schuiversnelheid; `s` is positief naar beneden.
- steunreacties aan schuiver en frame, om te bewaken dat de riem niet als geleiding wordt gebruikt.

De tekenconventie blijft dezelfde als in de vorige notebooks: openen betekent meestal `ds < 0`, en een negatieve `F_s` betekent dat de actuator omhoog moet trekken.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

results3_path = Path("notebook3_gravity_friction_results.npz")
if not results3_path.exists():
    raise FileNotFoundError("Run eerst Notebook 1, 2 en 3 zodat notebook3_gravity_friction_results.npz bestaat.")

required_keys = [
    "t", "s", "ds", "dds",
    "F_drive_s_inertia", "F_drive_s_gravity", "F_drive_s_total",
    "F_gravity_component", "F_friction_component", "F_slider_drive_component", "F_pin_friction_component",
    "P_act_total", "E_act_total",
    "hold_s_curve", "F_hold_s_curve", "R_Ax_hold_curve", "static_slider_capacity_curve",
    "R_Ax_total", "C_x_total", "C_y_total", "F_A_total_norm", "F_C_total_norm", "F_frame_total_norm",
    "A_max_full", "P_avg_full", "P_peak_full", "P_rms_full", "F_peak_full", "F_rms_full",
    "total_model_mass", "total_weight", "g",
    "mu_slider", "mu_pin", "pin_radius", "dyn_residual_total", "max_diff_notebook2",
]

data = np.load(results3_path, allow_pickle=True)
missing = [key for key in required_keys if key not in data.files]
if missing:
    raise KeyError(f"Ontbrekende keys in {results3_path}: {missing}")

t = data["t"]
s = data["s"]
ds = data["ds"]
dds = data["dds"]
Ts = float(np.median(np.diff(t))) if len(t) > 1 else 0.0
T_run = float(t[-1] - t[0])

F_drive_s_inertia = data["F_drive_s_inertia"]
F_drive_s_gravity = data["F_drive_s_gravity"]
F_drive_s_total = data["F_drive_s_total"]
F_gravity_component = data["F_gravity_component"]
F_friction_component = data["F_friction_component"]
F_slider_drive_component = data["F_slider_drive_component"]
F_pin_friction_component = data["F_pin_friction_component"]
P_act_total = data["P_act_total"]
E_act_total = data["E_act_total"]

hold_s_curve = data["hold_s_curve"]
F_hold_s_curve = data["F_hold_s_curve"]
R_Ax_hold_curve = data["R_Ax_hold_curve"]
static_slider_capacity_curve = data["static_slider_capacity_curve"]

R_Ax_total = data["R_Ax_total"]
C_x_total = data["C_x_total"]
C_y_total = data["C_y_total"]
F_A_total_norm = data["F_A_total_norm"]
F_C_total_norm = data["F_C_total_norm"]
F_frame_total_norm = data["F_frame_total_norm"]

total_model_mass = float(data["total_model_mass"])
total_weight = float(data["total_weight"])
g = float(data["g"])

motion_mask = np.abs(ds) > 1e-8
if not np.any(motion_mask):
    motion_mask = np.ones_like(ds, dtype=bool)
hold_mask = ~motion_mask

stroke = float(np.max(s) - np.min(s))
move_time = float(t[motion_mask][-1] - t[motion_mask][0]) if np.any(motion_mask) else T_run
active_time = max(move_time, Ts)

print("Data ingeladen uit Notebook 3:")
print(results3_path.resolve())
print(f"aantal tijdstappen                 : {len(t)}")
print(f"simulatieduur                      : {T_run:.3f} s")
print(f"effectieve bewegingstijd           : {active_time:.3f} s")
print(f"slag                               : {stroke:.4f} m")
print(f"totale bewegende modelmassa        : {total_model_mass:.3f} kg")
print(f"max |F_s| totaal, per mechanisme    : {np.max(np.abs(F_drive_s_total[motion_mask])):.3f} N")
print(f"max |F_hold|, per mechanisme        : {np.max(np.abs(F_hold_s_curve)):.3f} N")
print(f"validatie t.o.v. Notebook 2         : {float(data['max_diff_notebook2']):.3e} N")
print(f"max dynamisch residu Notebook 3     : {np.max(data['dyn_residual_total']):.3e}")


## Ontwerpkeuzes voor de aandrijving

De mechanische belasting wordt gesplitst in aandrijfkracht, houdkracht en steunreacties. Die splitsing bepaalt de aandrijfarchitectuur:

- de riem/kabel levert de verticale kracht langs `s`;
- de rem of vergrendeling houdt open, gesloten en tussenstanden vast;
- de schuivergeleiding neemt de zijreactie en het kantelmoment op;
- de controller, encoder en eindschakelaars zorgen voor herhaalbare positionering.

De notebook rekent daarom niet alleen een piekkracht uit, maar ook het nodige koppel, vermogen, remkoppel, poelietoerental, positiegevoeligheid en het effect van enkele ontwerpkeuzes.


In [ ]:
# ============================================================
# Instelbare aandrijfparameters
# ============================================================

drive_type = "belt_cable"       # praktische keuze: tandriem of kabel/riem langs de mast
mechanism_count = 1             # 1 = eenzijdig mechanisme, 2 = symmetrisch dubbel mechanisme

# Poelie/trommel aan de motorreductor-uitgang.
# De notebook kan automatisch een radius kiezen uit de kandidaatset.
auto_select_pulley = True
preferred_pulley_radius = 0.025  # [m] voorkeur: 25 mm als die binnen de limieten blijft
pulley_radius_candidates = np.array([0.015, 0.020, 0.025, 0.030, 0.040, 0.050, 0.060])
minimum_practical_pulley_radius = 0.020  # [m] kleine poelies vragen meer aandacht voor riembuiging
max_preferred_drive_torque = 12.0  # [Nm] gewenste bovengrens voor aandrijfkoppel aan uitgang
max_preferred_brake_torque = 8.0   # [Nm] gewenste bovengrens voor remkoppel aan uitgang
manual_pulley_radius = 0.025      # [m] alleen gebruikt als auto_select_pulley = False

drive_efficiency = 0.65           # [-] globale efficientie tussen motor/reductor en riemkracht
gear_efficiency = 0.75            # [-] orde-grootte voor reductiekast

# Veiligheidsfactoren
drive_safety_factor = 2.0         # voor actieve kracht/koppel/vermogen
brake_safety_factor = 2.0         # voor stilstand en tussenstanden

# Praktische minimale ontwerpkracht voor riem/kabel.
# Deze vloer voorkomt dat een theoretisch kleine kracht tot een te lichte buitenconstructie leidt.
line_force_floor_single = 200.0   # [N]
line_force_floor_double = 300.0   # [N]

# Motorconcept
motor_voltage = 24.0              # [V]
controller_current_margin = 1.50  # [-]
motor_nominal_speed_rpm = 3000.0  # [rpm] continu orde-grootte
motor_peak_speed_rpm = 6000.0     # [rpm] korte piek voor dit trage positioneersysteem

# Reductiekeuze: hoogste verhouding die de pieksnelheid nog toelaat geeft meer koppelreserve.
auto_select_gear_ratio = True
gear_ratio_candidates = np.array([25.0, 30.0, 40.0, 50.0, 60.0, 75.0, 100.0])
manual_gear_ratio = 60.0

# Gewenste uitgangssnelheid aan poelie/trommel.
target_output_rpm_min = 20.0      # [rpm]
target_output_rpm_max = 40.0      # [rpm]
allowable_peak_output_rpm = 120.0 # [rpm] korte piek aan poelie/reductoruitgang

# Precisie-inschatting
encoder_counts_per_motor_rev = 1024  # [counts/rev] eenvoudige encoder
encoder_decode_factor = 4            # quadrature decode
estimated_backlash_mm = 1.0          # [mm] conservatieve mechanische speling/elasticiteit
position_tolerance_mm = 5.0          # [mm] gewenste positioneernauwkeurigheid voor tussenstanden
effective_drive_stiffness = 2.0e5    # [N/m] riem + bevestiging + schuiver, ruwe orde-grootte

if drive_type != "belt_cable":
    raise ValueError("Deze notebook is uitgewerkt voor drive_type = 'belt_cable'.")
if mechanism_count not in (1, 2):
    raise ValueError("mechanism_count moet 1 of 2 zijn.")
if drive_efficiency <= 0 or gear_efficiency <= 0:
    raise ValueError("Efficienties moeten positief zijn.")
if drive_safety_factor < 1 or brake_safety_factor < 1:
    raise ValueError("Veiligheidsfactoren moeten minstens 1 zijn.")
if np.any(pulley_radius_candidates <= 0):
    raise ValueError("Alle kandidaatpoelieradii moeten positief zijn.")
if np.any(gear_ratio_candidates <= 0):
    raise ValueError("Alle kandidaat-reducties moeten positief zijn.")

line_force_floor = line_force_floor_single if mechanism_count == 1 else line_force_floor_double

print("Aandrijfparameters:")
print(f"drive_type                         : {drive_type}")
print(f"mechanism_count                    : {mechanism_count}")
print(f"auto_select_pulley                 : {auto_select_pulley}")
print(f"preferred_pulley_radius            : {preferred_pulley_radius*1000:.1f} mm")
print(f"max_preferred_drive_torque         : {max_preferred_drive_torque:.1f} Nm")
print(f"drive_efficiency                   : {drive_efficiency:.2f}")
print(f"drive_safety_factor                : {drive_safety_factor:.2f}")
print(f"brake_safety_factor                : {brake_safety_factor:.2f}")
print(f"praktische vloer lijnkracht         : {line_force_floor:.1f} N")
print(f"motor_voltage                      : {motor_voltage:.1f} V")


## Kracht, vermogen en koppel

De riemkracht is de kracht die nodig is om de schuiver langs `s` te bewegen. Voor een symmetrisch dubbel mechanisme wordt de verticale aandrijfkracht ongeveer verdubbeld, omdat beide zijden tegelijk geopend worden.

Voor de poelie geldt:

$$
T_{poelie}(t) = \frac{F_s(t)\,r}{\eta}
$$

met `r` de poelieradius en `eta` de globale aandrijfefficientie. Het mechanische vermogen aan de schuiver is:

$$
P_s(t) = F_s(t)\,\dot{s}(t)
$$

Voor openen zijn `F_s` en `ds` meestal allebei negatief, waardoor het geleverde vermogen positief wordt.


In [ ]:
F_s_one = F_drive_s_total
F_s_drive = mechanism_count * F_s_one
F_hold_drive_curve = mechanism_count * F_hold_s_curve

P_slider_drive = F_s_drive * ds
P_positive_slider = np.maximum(P_slider_drive, 0.0)
P_negative_slider = np.minimum(P_slider_drive, 0.0)

line_force_peak_motion = float(np.max(np.abs(F_s_drive[motion_mask])))
line_force_peak_hold = float(np.max(np.abs(F_hold_drive_curve)))
line_force_peak_operating = max(line_force_peak_motion, line_force_peak_hold)
line_force_design_raw = drive_safety_factor * line_force_peak_operating
line_force_design = max(line_force_design_raw, line_force_floor)

avg_line_speed = stroke / active_time


def pulley_metrics(radius):
    peak_rpm = np.max(np.abs(ds[motion_mask])) / (2.0 * np.pi * radius) * 60.0
    avg_rpm = avg_line_speed / (2.0 * np.pi * radius) * 60.0
    drive_torque = line_force_design * radius / drive_efficiency
    brake_torque = brake_safety_factor * line_force_peak_hold * radius
    mm_per_rev = 2.0 * np.pi * radius * 1000.0
    feasible = (
        (radius >= minimum_practical_pulley_radius)
        and (peak_rpm <= allowable_peak_output_rpm)
        and (drive_torque <= max_preferred_drive_torque)
        and (brake_torque <= max_preferred_brake_torque)
    )
    return avg_rpm, peak_rpm, drive_torque, brake_torque, mm_per_rev, feasible


radius_rows = []
for radius in pulley_radius_candidates:
    radius_rows.append((radius, *pulley_metrics(radius)))
radius_rows = np.array(radius_rows, dtype=float)

candidate_r = radius_rows[:, 0]
candidate_feasible = radius_rows[:, 6].astype(bool)
preferred_matches = np.isclose(candidate_r, preferred_pulley_radius)

if auto_select_pulley:
    if np.any(candidate_feasible & preferred_matches):
        selected_idx = int(np.where(candidate_feasible & preferred_matches)[0][0])
        pulley_selection_note = "voorkeursradius is haalbaar"
    elif np.any(candidate_feasible):
        feasible_indices = np.where(candidate_feasible)[0]
        selected_idx = int(feasible_indices[np.argmin(np.abs(candidate_r[feasible_indices] - preferred_pulley_radius))])
        pulley_selection_note = "voorkeursradius niet haalbaar; dichtstbij haalbare kandidaat gekozen"
    else:
        # Kies de kandidaat met de kleinste relatieve overschrijding van pieksnelheid en minimumradius.
        rpm_violation = np.maximum(radius_rows[:, 2] / allowable_peak_output_rpm - 1.0, 0.0)
        radius_violation = np.maximum(minimum_practical_pulley_radius / candidate_r - 1.0, 0.0)
        selected_idx = int(np.argmin(rpm_violation + radius_violation))
        pulley_selection_note = "geen kandidaat voldoet volledig; minst slechte kandidaat gekozen"
else:
    manual_matches = np.isclose(candidate_r, manual_pulley_radius)
    if np.any(manual_matches):
        selected_idx = int(np.where(manual_matches)[0][0])
    else:
        selected_idx = int(np.argmin(np.abs(candidate_r - manual_pulley_radius)))
    pulley_selection_note = "handmatige radius gebruikt"

drive_pulley_radius = float(radius_rows[selected_idx, 0])
avg_equiv_output_rpm = float(radius_rows[selected_idx, 1])
peak_output_rpm = float(radius_rows[selected_idx, 2])
pulley_radius_feasible = bool(radius_rows[selected_idx, 6])

pulley_speed_rps = ds / (2.0 * np.pi * drive_pulley_radius)
pulley_speed_rpm = 60.0 * pulley_speed_rps
T_pulley_active = F_s_drive * drive_pulley_radius / drive_efficiency
T_pulley_peak_active = float(np.max(np.abs(T_pulley_active[motion_mask])))
T_pulley_design = line_force_design * drive_pulley_radius / drive_efficiency

P_peak_slider = float(np.max(P_positive_slider[motion_mask]))
P_rms_slider = float(np.sqrt(np.mean(P_slider_drive[motion_mask]**2)))
P_peak_motor_input = drive_safety_factor * P_peak_slider / drive_efficiency
P_rms_motor_input = drive_safety_factor * P_rms_slider / drive_efficiency
I_peak_est = controller_current_margin * P_peak_motor_input / motor_voltage
I_rms_est = controller_current_margin * P_rms_motor_input / motor_voltage

E_positive_slider = float(np.trapezoid(P_positive_slider, t))
E_negative_slider = float(np.trapezoid(-P_negative_slider, t))
E_net_slider = float(np.trapezoid(P_slider_drive, t))

P_avg_drive = float(np.mean(P_slider_drive))
A_theta_drive = np.cumsum((P_slider_drive - P_avg_drive) * Ts)
A_max_drive = float(np.max(A_theta_drive) - np.min(A_theta_drive))

if auto_select_gear_ratio:
    gear_feasible = gear_ratio_candidates * peak_output_rpm <= motor_peak_speed_rpm
    if np.any(gear_feasible):
        selected_gear_ratio = float(np.max(gear_ratio_candidates[gear_feasible]))
        gear_selection_note = "hoogste haalbare reductie binnen motorpieksnelheid"
    else:
        selected_gear_ratio = float(np.min(gear_ratio_candidates))
        gear_selection_note = "geen reductie voldoet volledig aan motorpieksnelheid"
else:
    selected_gear_ratio = float(manual_gear_ratio)
    gear_selection_note = "handmatige reductie gebruikt"

motor_speed_peak_est = peak_output_rpm * selected_gear_ratio
motor_speed_avg_est = avg_equiv_output_rpm * selected_gear_ratio
motor_torque_design_est = T_pulley_design / (selected_gear_ratio * gear_efficiency)

print("Aandrijfkracht en motorbelasting:")
print(f"piek lijnkracht tijdens beweging     : {line_force_peak_motion:.2f} N")
print(f"piek houdkracht over alle standen    : {line_force_peak_hold:.2f} N")
print(f"ontwerpkracht zonder praktische vloer : {line_force_design_raw:.2f} N")
print(f"gekozen ontwerplijnkracht            : {line_force_design:.2f} N")
print()
print("Poelie- en reductiekeuze:")
print(f"gekozen poelieradius                 : {drive_pulley_radius*1000:.1f} mm ({pulley_selection_note})")
print(f"gemiddeld uitgangstoerental          : {avg_equiv_output_rpm:.1f} rpm")
print(f"piek-uitgangstoerental               : {peak_output_rpm:.1f} rpm")
print(f"gekozen reductie                     : {selected_gear_ratio:.0f}:1 ({gear_selection_note})")
print(f"geschatte motorpieksnelheid          : {motor_speed_peak_est:.0f} rpm")
print(f"geschatte gemiddelde motorsnelheid   : {motor_speed_avg_est:.0f} rpm")
print()
print(f"piek aandrijfkoppel incl. verliezen  : {T_pulley_peak_active:.3f} Nm")
print(f"ontwerp aandrijfkoppel incl. verlies : {T_pulley_design:.3f} Nm")
print(f"geschat motorkoppel voor reductie    : {motor_torque_design_est:.3f} Nm")
print(f"piek positief schuiververmogen       : {P_peak_slider:.2f} W")
print(f"piek motor-ingangsvermogen ontwerp   : {P_peak_motor_input:.2f} W")
print(f"RMS motor-ingangsvermogen ontwerp    : {P_rms_motor_input:.2f} W")
print(f"geschatte piekstroom bij 24 V        : {I_peak_est:.2f} A")
print(f"geschatte RMS-stroom bij 24 V        : {I_rms_est:.2f} A")
print()
print(f"positieve mechanische arbeid         : {E_positive_slider:.2f} J")
print(f"netto mechanische arbeid             : {E_net_slider:.2f} J")
print(f"arbeids-surplus aandrijving          : {A_max_drive:.2f} J")

if not pulley_radius_feasible:
    print("Waarschuwing: de gekozen poelieradius voldoet niet volledig aan de ingestelde grenzen.")
if motor_speed_peak_est > motor_peak_speed_rpm:
    print("Waarschuwing: de gekozen reductie vraagt meer motorpieksnelheid dan ingesteld.")

fig_load, ax_load = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
fig_load.suptitle("Belasting van de riem-/motoraandrijving")

ax_load[0, 0].plot(t, F_s_drive, label="F_s")
ax_load[0, 0].axhline(line_force_design, color="tab:red", ls="--", label="ontwerpkracht")
ax_load[0, 0].axhline(-line_force_design, color="tab:red", ls="--")
ax_load[0, 0].set_title("Lijnkracht in de aandrijving")
ax_load[0, 0].set_xlabel("t [s]"); ax_load[0, 0].set_ylabel("N")
ax_load[0, 0].grid(True); ax_load[0, 0].legend()

ax_load[0, 1].plot(t, T_pulley_active, label="T actief")
ax_load[0, 1].axhline(T_pulley_design, color="tab:red", ls="--", label="ontwerpkoppel")
ax_load[0, 1].axhline(-T_pulley_design, color="tab:red", ls="--")
ax_load[0, 1].set_title("Aandrijfkoppel inclusief verliezen")
ax_load[0, 1].set_xlabel("t [s]"); ax_load[0, 1].set_ylabel("Nm")
ax_load[0, 1].grid(True); ax_load[0, 1].legend()

ax_load[1, 0].plot(t, P_slider_drive, label="P schuiver")
ax_load[1, 0].axhline(0.0, color="black", lw=0.8)
ax_load[1, 0].set_title("Mechanisch vermogen aan de schuiver")
ax_load[1, 0].set_xlabel("t [s]"); ax_load[1, 0].set_ylabel("W")
ax_load[1, 0].grid(True); ax_load[1, 0].legend()

ax_load[1, 1].plot(t, pulley_speed_rpm, label="poelie")
ax_load[1, 1].axhline(target_output_rpm_max, color="tab:orange", ls="--", label="richtwaarde continu")
ax_load[1, 1].axhline(-target_output_rpm_max, color="tab:orange", ls="--")
ax_load[1, 1].axhline(allowable_peak_output_rpm, color="tab:red", ls=":", label="toelaatbare piek")
ax_load[1, 1].axhline(-allowable_peak_output_rpm, color="tab:red", ls=":")
ax_load[1, 1].set_title("Poelietoerental")
ax_load[1, 1].set_xlabel("t [s]"); ax_load[1, 1].set_ylabel("rpm")
ax_load[1, 1].grid(True); ax_load[1, 1].legend()

plt.show()


## Houdkracht, rem en tussenstanden

Een riem houdt de paraplu niet vanzelf vast. De tussenstand wordt vastgehouden door de motorregeling, een zelfremmende overbrenging of beter: een echte rem/vergrendeling.

Voor de rem op de poelie-/reductoruitgang wordt gerekend met:

$$
T_{rem} \ge SF_{rem}\, |F_{hold}|\, r
$$

Er wordt bewust niet gerekend op toevallige schuiverwrijving. Die wrijving kan veranderen door slijtage, water, vuil, temperatuur of smering.


In [ ]:
T_hold_output_curve = np.abs(F_hold_drive_curve) * drive_pulley_radius
T_hold_brake_design_curve = brake_safety_factor * T_hold_output_curve
T_hold_motor_shaft_curve = T_hold_brake_design_curve / (selected_gear_ratio * gear_efficiency)
T_hold_motor_conservative_curve = brake_safety_factor * np.abs(F_hold_drive_curve) * drive_pulley_radius / drive_efficiency

half_s = 0.5 * (np.min(hold_s_curve) + np.max(hold_s_curve))
i_open = int(np.argmin(hold_s_curve))
i_closed = int(np.argmax(hold_s_curve))
i_half = int(np.argmin(np.abs(hold_s_curve - half_s)))
i_hold_peak = int(np.argmax(np.abs(F_hold_drive_curve)))

print("Houdanalyse voor tussenstanden:")
for label, idx in [("open", i_open), ("halfopen", i_half), ("gesloten", i_closed), ("max", i_hold_peak)]:
    print(
        f"{label:9s} | s = {hold_s_curve[idx]:.3f} m | "
        f"|F_hold| = {abs(F_hold_drive_curve[idx]):.2f} N | "
        f"T_rem uitgang = {T_hold_brake_design_curve[idx]:.3f} Nm | "
        f"T_rem motoras ~ {T_hold_motor_shaft_curve[idx]:.3f} Nm"
    )

print()
print(f"max remkoppel aan poelie-/reductoruitgang : {np.max(T_hold_brake_design_curve):.3f} Nm")
print(f"max remkoppel omgerekend naar motoras     : {np.max(T_hold_motor_shaft_curve):.3f} Nm")
print(f"conservatief houdkoppel incl. verliezen   : {np.max(T_hold_motor_conservative_curve):.3f} Nm")
print(f"max theoretische schuiverwrijvingsgrens   : {np.max(static_slider_capacity_curve)*mechanism_count:.2f} N")
print("Conclusie: gebruik een rem/vergrendeling; ontwerp niet op schuiverwrijving alleen.")

fig_hold, ax_hold = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig_hold.suptitle("Houdkracht en remdimensionering")

ax_hold[0].plot(hold_s_curve, F_hold_drive_curve, label="F_hold")
ax_hold[0].axhline(0.0, color="black", lw=0.8)
ax_hold[0].set_title("Statische houdkracht")
ax_hold[0].set_xlabel("s [m]"); ax_hold[0].set_ylabel("N")
ax_hold[0].grid(True); ax_hold[0].legend()

ax_hold[1].plot(hold_s_curve, T_hold_brake_design_curve, label="uitgang")
ax_hold[1].plot(hold_s_curve, T_hold_motor_shaft_curve, "--", label="motoras, benaderd")
ax_hold[1].set_title("Benodigd remkoppel")
ax_hold[1].set_xlabel("s [m]"); ax_hold[1].set_ylabel("Nm")
ax_hold[1].grid(True); ax_hold[1].legend()

plt.show()


## Poelieradius en motorselectie

De poelieradius geeft een duidelijke afweging:

- kleinere radius: minder koppel nodig, maar hoger toerental en meer riembuiging;
- grotere radius: lager toerental en meer verplaatsing per omwenteling, maar meer koppel nodig;
- voor precisie geeft een kleinere radius een kleinere verplaatsing per omwenteling, maar de echte nauwkeurigheid wordt meestal beperkt door speling, riemrek en schuivergeleiding.

De notebook kiest automatisch een radius uit `pulley_radius_candidates`. De voorkeur blijft 25 mm als die binnen de ingestelde grenzen past. Als het traject of de geometrie wijzigt en die radius niet meer haalbaar is, kiest de notebook de dichtstbijzijnde haalbare kandidaat.


In [ ]:
print("Invloed van poelieradius:")
print("r [mm] | avg rpm | peak rpm | T_drive incl. verlies [Nm] | T_rem uitgang [Nm] | mm/rev | status")
for radius, rpm_avg, rpm_peak, T_design, T_hold_design, mm_per_rev, feasible in radius_rows:
    status = "gekozen" if np.isclose(radius, drive_pulley_radius) else ("haalbaar" if feasible else "niet haalbaar")
    print(
        f"{1000*radius:6.1f} | {rpm_avg:7.1f} | {rpm_peak:8.1f} | "
        f"{T_design:26.2f} | {T_hold_design:16.2f} | {mm_per_rev:6.1f} | {status}"
    )

gear_rows = []
for ratio in gear_ratio_candidates:
    motor_peak = ratio * peak_output_rpm
    motor_avg = ratio * avg_equiv_output_rpm
    motor_torque = T_pulley_design / (ratio * gear_efficiency)
    feasible = motor_peak <= motor_peak_speed_rpm
    gear_rows.append((ratio, motor_avg, motor_peak, motor_torque, feasible))
gear_rows = np.array(gear_rows, dtype=float)

print()
print("Invloed van reductieverhouding:")
print("ratio | motor avg rpm | motor peak rpm | motor torque ontwerp [Nm] | status")
for ratio, motor_avg, motor_peak, motor_torque, feasible in gear_rows:
    status = "gekozen" if np.isclose(ratio, selected_gear_ratio) else ("haalbaar" if feasible else "te snel")
    print(f"{ratio:5.0f} | {motor_avg:13.0f} | {motor_peak:14.0f} | {motor_torque:24.3f} | {status}")

fig_radius, ax_radius = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fig_radius.suptitle("Poelie- en reductiekeuze")

radius_mm = radius_rows[:, 0] * 1000.0
selected_radius_mm = drive_pulley_radius * 1000.0

ax_radius[0].plot(radius_mm, radius_rows[:, 2], "o-", label="piek rpm")
ax_radius[0].axhline(allowable_peak_output_rpm, color="tab:red", ls="--", label="limiet")
ax_radius[0].axvline(selected_radius_mm, color="black", ls=":", label="gekozen")
ax_radius[0].set_xlabel("r [mm]"); ax_radius[0].set_ylabel("rpm")
ax_radius[0].set_title("Poelietoerental")
ax_radius[0].grid(True); ax_radius[0].legend()

ax_radius[1].plot(radius_mm, radius_rows[:, 3], "o-", label="aandrijfkoppel")
ax_radius[1].plot(radius_mm, radius_rows[:, 4], "s--", label="remkoppel")
ax_radius[1].axvline(selected_radius_mm, color="black", ls=":", label="gekozen")
ax_radius[1].set_xlabel("r [mm]"); ax_radius[1].set_ylabel("Nm")
ax_radius[1].set_title("Koppel")
ax_radius[1].grid(True); ax_radius[1].legend()

ax_radius[2].plot(gear_rows[:, 0], gear_rows[:, 2], "o-", label="motor piek rpm")
ax_radius[2].axhline(motor_peak_speed_rpm, color="tab:red", ls="--", label="motor piekgrens")
ax_radius[2].axvline(selected_gear_ratio, color="black", ls=":", label="gekozen")
ax_radius[2].set_xlabel("reductie i [-]"); ax_radius[2].set_ylabel("rpm")
ax_radius[2].set_title("Reductie en motorsnelheid")
ax_radius[2].grid(True); ax_radius[2].legend()

plt.show()


## Precisie van de aandrijving

Voor precisie zijn drie niveaus belangrijk:

1. **Meetresolutie:** encoder op motor of uitgangsas.
2. **Transmissie:** riemspanning, reductiespeling en poelieradius.
3. **Mechanische structuur:** stijve schuivergeleiding, zodat de zijreactie geen kanteling of blokkering veroorzaakt.

De theoretische encoderresolutie is meestal veel kleiner dan de echte mechanische fout. Daarom is een stijve, spelingsarme schuiver/collar belangrijker dan alleen een hogere encoderresolutie.


In [ ]:
line_per_output_rev = 2.0 * np.pi * drive_pulley_radius
encoder_counts_effective = encoder_counts_per_motor_rev * encoder_decode_factor
linear_resolution_motor_encoder = line_per_output_rev / (selected_gear_ratio * encoder_counts_effective)
linear_resolution_output_encoder = line_per_output_rev / encoder_counts_effective

elastic_deflection_peak = line_force_peak_operating / effective_drive_stiffness
elastic_deflection_design = line_force_design / effective_drive_stiffness

print("Precisie-inschatting:")
print(f"verplaatsing per poelieomwenteling        : {line_per_output_rev*1000:.2f} mm/rev")
print(f"resolutie motorencoder na reductie        : {linear_resolution_motor_encoder*1000:.4f} mm/count")
print(f"resolutie encoder direct op uitgang        : {linear_resolution_output_encoder*1000:.4f} mm/count")
print(f"geschatte mechanische speling/elasticiteit : {estimated_backlash_mm:.2f} mm")
print(f"elastische verplaatsing bij piekbelasting  : {elastic_deflection_peak*1000:.3f} mm")
print(f"elastische verplaatsing bij ontwerpkracht  : {elastic_deflection_design*1000:.3f} mm")
print(f"gewenste positioneernauwkeurigheid         : {position_tolerance_mm:.2f} mm")

precision_ok = estimated_backlash_mm + elastic_deflection_peak * 1000.0 <= position_tolerance_mm
if precision_ok:
    print("Conclusie: de orde-grootte is voldoende voor paraplupositionering, mits de schuivergeleiding spelingsarm is.")
else:
    print("Conclusie: de mechanische speling/stijfheid wordt kritischer dan encoderresolutie.")

fig_precision, ax_precision = plt.subplots(1, 1, figsize=(8, 4), constrained_layout=True)
fig_precision.suptitle("Precisiebronnen")

labels = ["motorencoder", "uitgangsencoder", "speling", "riemrek piek", "riemrek ontwerp"]
values_mm = [
    linear_resolution_motor_encoder * 1000.0,
    linear_resolution_output_encoder * 1000.0,
    estimated_backlash_mm,
    elastic_deflection_peak * 1000.0,
    elastic_deflection_design * 1000.0,
]
ax_precision.bar(labels, values_mm)
ax_precision.axhline(position_tolerance_mm, color="tab:red", ls="--", label="tolerantie")
ax_precision.set_ylabel("mm")
ax_precision.grid(True, axis="y")
ax_precision.legend()
plt.xticks(rotation=20)
plt.show()


## Energieverbruik en trajectkeuze

Een bewegingswet met hogere versnelling of ruk vraagt meer dynamische belasting. Voor deze paraplu is de inertiecomponent echter klein ten opzichte van zwaartekracht en wrijving. Daardoor verlaagt trager bewegen vooral het piekvermogen en de rustiger regeling, maar niet de zwaartekrachtarbeid zelf.

De volgende cel maakt een eenvoudige schaalanalyse: hetzelfde pad wordt sneller of trager afgelegd. De inertiecomponent wordt benaderd als evenredig met `1/tijd_schaal^2`; zwaartekracht en Coulombachtige wrijving blijven ongeveer dezelfde krachtcomponent. Dit is geen vervanging van Notebook 3, maar een compacte ontwerpcheck.


In [ ]:
time_scale_factors = np.array([0.60, 0.80, 1.00, 1.25, 1.50, 2.00])
non_inertial_component = F_s_drive - mechanism_count * F_drive_s_inertia

scale_rows = []
for lam in time_scale_factors:
    # lam > 1 betekent trager; lam < 1 betekent sneller.
    F_scaled = non_inertial_component + mechanism_count * F_drive_s_inertia / lam**2
    ds_scaled = ds / lam
    t_scaled = (t - t[0]) * lam + t[0]
    P_scaled = F_scaled * ds_scaled
    E_pos_scaled = np.trapezoid(np.maximum(P_scaled, 0.0), t_scaled)
    P_peak_scaled = np.max(np.maximum(P_scaled[motion_mask], 0.0))
    F_peak_scaled = np.max(np.abs(F_scaled[motion_mask]))
    A_theta_scaled = np.cumsum((P_scaled - np.mean(P_scaled)) * Ts * lam)
    A_max_scaled = np.max(A_theta_scaled) - np.min(A_theta_scaled)
    scale_rows.append((lam, active_time * lam, F_peak_scaled, P_peak_scaled, E_pos_scaled, A_max_scaled))

print("Trajectschaal-trade-off:")
print("factor | beweegtijd [s] | peak |F_s| [N] | peak P [W] | positieve arbeid [J] | A_max [J]")
for lam, T_move_scaled, F_peak_scaled, P_peak_scaled, E_pos_scaled, A_max_scaled in scale_rows:
    print(f"{lam:5.2f} | {T_move_scaled:13.2f} | {F_peak_scaled:12.2f} | {P_peak_scaled:10.2f} | {E_pos_scaled:18.2f} | {A_max_scaled:8.2f}")

fig_scale, ax_scale = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fig_scale.suptitle("Effect van sneller/trager traject")

ax_scale[0].plot(time_scale_factors, [row[2] for row in scale_rows], "o-")
ax_scale[0].set_xlabel("tijd_schaal [-]"); ax_scale[0].set_ylabel("N")
ax_scale[0].set_title("Piek kracht")
ax_scale[0].grid(True)

ax_scale[1].plot(time_scale_factors, [row[3] for row in scale_rows], "o-")
ax_scale[1].set_xlabel("tijd_schaal [-]"); ax_scale[1].set_ylabel("W")
ax_scale[1].set_title("Piek vermogen")
ax_scale[1].grid(True)

ax_scale[2].plot(time_scale_factors, [row[4] for row in scale_rows], "o-", label="positieve arbeid")
ax_scale[2].plot(time_scale_factors, [row[5] for row in scale_rows], "s--", label="A_max")
ax_scale[2].set_xlabel("tijd_schaal [-]"); ax_scale[2].set_ylabel("J")
ax_scale[2].set_title("Arbeid en arbeids-surplus")
ax_scale[2].grid(True); ax_scale[2].legend()

plt.show()


## Krachtgeneratie en symmetrische uitvoering

Voor meer kracht kan men een sterkere motor, grotere reductie of kleinere poelie kiezen. Maar de motor is niet de enige ontwerpfactor. De grote zijreactie aan de schuiver moet door de geleiding en mast opgenomen worden, niet door de riem.

Een symmetrische dubbele uitvoering rond de mast is mechanisch aantrekkelijk: de verticale aandrijfkracht wordt ongeveer dubbel, maar horizontale reacties kunnen elkaar grotendeels opheffen als de geometrie echt gespiegeld en stijf verbonden is. Dat verlaagt vooral lokale mastbelasting en kanteling, niet de benodigde verticale arbeid.


In [ ]:
counts = np.array([1, 2])
comparison_rows = []
for count in counts:
    floor = line_force_floor_single if count == 1 else line_force_floor_double
    F_peak_motion_count = count * np.max(np.abs(F_drive_s_total[motion_mask]))
    F_peak_hold_count = count * np.max(np.abs(F_hold_s_curve))
    F_peak_operating_count = max(F_peak_motion_count, F_peak_hold_count)
    F_design_count = max(drive_safety_factor * F_peak_operating_count, floor)
    T_design_count = F_design_count * drive_pulley_radius / drive_efficiency
    P_peak_count = count * P_peak_slider
    P_input_design_count = drive_safety_factor * P_peak_count / drive_efficiency
    comparison_rows.append((count, F_peak_operating_count, F_design_count, T_design_count, P_input_design_count))

# Lokale zijreactie uit Notebook 3 voor een enkel mechanisme.
side_reaction_peak = float(np.max(np.abs(R_Ax_total)))
local_bending_est = float(np.max(np.abs(R_Ax_total * s)))

print("Vergelijking eenzijdig versus symmetrisch dubbel:")
print("count | peak operation force [N] | design line force [N] | design torque [Nm] | design peak input P [W]")
for row in comparison_rows:
    print(f"{row[0]:5d} | {row[1]:24.2f} | {row[2]:21.2f} | {row[3]:18.2f} | {row[4]:21.2f}")

print()
print("Geleiding en mastbelasting:")
print(f"max lokale zijreactie schuiver A_x, enkel mechanisme : {side_reaction_peak:.2f} N")
print(f"ruwe momentarm-check max |A_x * s|                 : {local_bending_est:.2f} Nm")
print("Bij een symmetrische dubbele uitvoering kan de globale horizontale resultante dalen,")
print("maar elke schuiver/geleider moet lokaal nog steeds zijn eigen zijreactie dragen.")

fig_count, ax_count = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
fig_count.suptitle("Eenzijdig versus symmetrisch dubbel")

ax_count[0].bar([str(row[0]) for row in comparison_rows], [row[2] for row in comparison_rows])
ax_count[0].set_xlabel("aantal mechanismen")
ax_count[0].set_ylabel("N")
ax_count[0].set_title("Ontwerplijnkracht")
ax_count[0].grid(True, axis="y")

ax_count[1].bar([str(row[0]) for row in comparison_rows], [row[3] for row in comparison_rows])
ax_count[1].set_xlabel("aantal mechanismen")
ax_count[1].set_ylabel("Nm")
ax_count[1].set_title("Ontwerpkoppel")
ax_count[1].grid(True, axis="y")

plt.show()


## Ontwerpconclusie

De motor is niet bedoeld om de schuiver zijdelings te geleiden. De correcte taakverdeling is:

- **motor + reductor + riem/kabel:** verticale aandrijfkracht en positionering;
- **rem/vergrendeling:** veilig vasthouden in open, gesloten en tussenstanden;
- **schuiver/collar + mastgeleiding:** horizontale steunreacties en kantelmomenten;
- **controller + encoder + eindschakelaars:** precisie, homing en eindpositiebeveiliging.

Een vliegwiel is hier niet de eerste logische keuze, omdat de paraplu een traag positioneermechanisme met stilstand in willekeurige tussenstanden is. Voor energie-opslag of zwaartekrachtcompensatie is een trekveer/gasveer later logischer dan een vliegwiel, maar die veerassistentie hoort in een aparte uitbreiding zodat de motorbasis zuiver blijft.


In [ ]:
recommended_motor_power_floor = 50.0 if mechanism_count == 1 else 100.0
recommended_motor_power_peak = max(recommended_motor_power_floor, 1.5 * P_peak_motor_input)
recommended_output_torque = max(T_pulley_design, 8.0 if mechanism_count == 1 else 12.0)
recommended_brake_torque = max(np.max(T_hold_brake_design_curve), 2.0)

results4_path = Path("notebook4_aandrijving_results.npz").resolve()
np.savez(
    results4_path,
    t=t, s=s, ds=ds, dds=dds,
    drive_type=np.array(drive_type),
    mechanism_count=mechanism_count,
    auto_select_pulley=auto_select_pulley,
    preferred_pulley_radius=preferred_pulley_radius,
    drive_pulley_radius=drive_pulley_radius,
    pulley_radius_feasible=pulley_radius_feasible,
    pulley_selection_note=np.array(pulley_selection_note),
    max_preferred_drive_torque=max_preferred_drive_torque,
    max_preferred_brake_torque=max_preferred_brake_torque,
    drive_efficiency=drive_efficiency,
    gear_efficiency=gear_efficiency,
    selected_gear_ratio=selected_gear_ratio,
    gear_selection_note=np.array(gear_selection_note),
    motor_speed_peak_est=motor_speed_peak_est,
    motor_speed_avg_est=motor_speed_avg_est,
    motor_torque_design_est=motor_torque_design_est,
    drive_safety_factor=drive_safety_factor,
    brake_safety_factor=brake_safety_factor,
    line_force_peak_motion=line_force_peak_motion,
    line_force_peak_hold=line_force_peak_hold,
    line_force_peak_operating=line_force_peak_operating,
    line_force_design=line_force_design,
    T_pulley_active=T_pulley_active,
    T_pulley_peak_active=T_pulley_peak_active,
    T_pulley_design=T_pulley_design,
    P_slider_drive=P_slider_drive,
    P_peak_slider=P_peak_slider,
    P_rms_slider=P_rms_slider,
    P_peak_motor_input=P_peak_motor_input,
    P_rms_motor_input=P_rms_motor_input,
    I_peak_est=I_peak_est,
    I_rms_est=I_rms_est,
    E_positive_slider=E_positive_slider,
    E_net_slider=E_net_slider,
    A_theta_drive=A_theta_drive,
    A_max_drive=A_max_drive,
    pulley_speed_rpm=pulley_speed_rpm,
    avg_equiv_output_rpm=avg_equiv_output_rpm,
    peak_output_rpm=peak_output_rpm,
    hold_s_curve=hold_s_curve,
    F_hold_drive_curve=F_hold_drive_curve,
    T_hold_output_curve=T_hold_output_curve,
    T_hold_brake_design_curve=T_hold_brake_design_curve,
    T_hold_motor_shaft_curve=T_hold_motor_shaft_curve,
    side_reaction_peak=side_reaction_peak,
    local_bending_est=local_bending_est,
    radius_options=pulley_radius_candidates,
    radius_rows=np.array(radius_rows, dtype=float),
    gear_ratio_candidates=gear_ratio_candidates,
    gear_rows=np.array(gear_rows, dtype=float),
    time_scale_factors=time_scale_factors,
    scale_rows=np.array(scale_rows, dtype=float),
    comparison_rows=np.array(comparison_rows, dtype=float),
    linear_resolution_motor_encoder=linear_resolution_motor_encoder,
    linear_resolution_output_encoder=linear_resolution_output_encoder,
    elastic_deflection_peak=elastic_deflection_peak,
    elastic_deflection_design=elastic_deflection_design,
    recommended_motor_power_peak=recommended_motor_power_peak,
    recommended_output_torque=recommended_output_torque,
    recommended_brake_torque=recommended_brake_torque,
)

print("Notebook 4-resultaten opgeslagen in:")
print(results4_path)
print()
print("SAMENVATTING - AANDRIJVING")
print("=" * 56)
print(f"voorkeursconcept                  : 24 V DC/BLDC reductiemotor + encoder + rem")
print(f"aandrijving                       : gesloten tandriem/kabel langs mast")
print(f"gekozen poelieradius              : {drive_pulley_radius*1000:.1f} mm")
print(f"gekozen reductie                  : {selected_gear_ratio:.0f}:1")
print(f"piek lijnkracht, operationeel      : {line_force_peak_operating:.2f} N")
print(f"gekozen ontwerplijnkracht          : {line_force_design:.2f} N")
print(f"ontwerp aandrijfkoppel incl. verlies: {T_pulley_design:.2f} Nm")
print(f"max remkoppel aan uitgang          : {np.max(T_hold_brake_design_curve):.2f} Nm")
print(f"max remkoppel aan motoras          : {np.max(T_hold_motor_shaft_curve):.3f} Nm")
print(f"piek motor-ingangsvermogen ontwerp : {P_peak_motor_input:.2f} W")
print(f"aanbevolen motorvermogen praktisch : {recommended_motor_power_peak:.1f} W klasse")
print(f"aanbevolen uitgangskoppel praktisch: {recommended_output_torque:.1f} Nm klasse")
print(f"vereist piek-uitgangstoerental     : {peak_output_rpm:.1f} rpm")
print(f"gemiddeld equivalent toerental     : {avg_equiv_output_rpm:.1f} rpm")
print(f"geschatte motorpieksnelheid        : {motor_speed_peak_est:.0f} rpm")
print(f"max lokale zijreactie schuiver      : {side_reaction_peak:.2f} N")
print(f"arbeids-surplus aandrijving         : {A_max_drive:.2f} J")
print()
print("Kernadvies:")
print("Gebruik een motor met encoder en rem. Laat de riem alleen de verticale aandrijfkracht leveren.")
print("Dimensioneer de schuivergeleiding apart op de grote zijreactie; die hoort niet in de riem.")
